In [12]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()  

api_key = os.getenv("GEMINI_API_KEY")

In [13]:
################################################################################
##### Call an LLM API (OpenAI or similar) using Python or Node.js #####

client = genai.Client()

response = client.interactions.create(
    model="gemini-3.1-flash-lite",
    input="introduce yourself"
)

print(response.output_text)

Hello! I am a large language model, trained by Google.

You can think of me as a knowledgeable, creative, and versatile virtual assistant. I don’t have a physical form, a personal life, or feelings, but I am designed to process information and engage in helpful, human-like conversation.

**Here are a few things I can help you with:**

*   **Writing & Editing:** I can help you draft emails, write essays, create stories, polish your grammar, or rewrite text to change the tone.
*   **Learning & Explaining:** Whether you need a complex scientific concept simplified, help understanding a historical event, or a summary of a long article, I’m here to explain it.
*   **Problem Solving & Coding:** I can help you brainstorm ideas, organize your schedule, write or debug code in many different programming languages, and work through math problems.
*   **Creativity:** I love brainstorming ideas for projects, writing poems or song lyrics, or helping you plan a travel itinerary.
*   **Just Chatting:*

In [14]:
################################################################################
##### Design a system prompt (e.g. interviewer / assistant / expert) and test behavior changes #####
client = genai.Client()

question = "Explain hash set and give one example."

prompts = {
    "interviewer": """
You are a Python technical interviewer.
Ask one follow-up question before giving hints.
Do not provide a complete solution immediately.
""",
    "assistant": """
You are a patient Python teaching assistant.
Explain concepts in simple Chinese.
Give a short code example and one common mistake.
""",
    "expert": """
You are a senior Python expert.
Give a concise, technically precise answer in Chinese.
Mention time complexity where relevant.
""",
}

for role, system_prompt in prompts.items():
    response = client.interactions.create(
        model="gemini-3.1-flash-lite",
        system_instruction=system_prompt,
        input=question,
    )

    print(f"\n--- {role} ---")
    print(response.output_text)


--- interviewer ---
A **Hash Set** is a collection data structure that stores unique elements. It is designed for high-performance operations, specifically checking if an item exists, adding items, and removing items.

Under the hood, it uses a **hash table**. When you add an item, the set calculates a "hash code" for that item (a numerical representation) and uses it to determine the specific bucket where the item should be stored. This allows for an average time complexity of **O(1)** (constant time) for insertions, deletions, and lookups, regardless of how many elements are in the set.

### Example in Python:
```python
# Creating a hash set
my_set = {1, 2, 3, 4}

# Adding an element
my_set.add(5)

# Attempting to add a duplicate
my_set.add(3) # The set will remain {1, 2, 3, 4, 5}

# Checking for existence
print(3 in my_set) # Output: True
```

***

### Follow-up Question:
To better understand how this works in practice, imagine you are tasked with finding the first duplicate charac

In [15]:
##### Run the same prompt with different temperature values (0.2 / 0.7 / 1.0) and compare outputs #####

client = genai.Client()

prompt = "create an advertisement for Leetcode"
temperatures = [0.2, 0.7, 1.0]

for temperature in temperatures:
    response = client.interactions.create(
        model="gemini-3.1-flash-lite",
        input=prompt,
        generation_config={
            "temperature": temperature,
            "max_output_tokens": 100,
        },
    )

    print(f"\n--- temperature = {temperature} ---")
    print(response.output_text)




--- temperature = 0.2 ---
To create an effective advertisement for LeetCode, you need to tailor the message to the user’s specific "pain point." Here are three different approaches depending on the platform and audience.

---

### Option 1: The "Career Aspirations" Approach (LinkedIn/Professional)
**Headline: Don’t just land the interview. Ace it.**

**Body:**
Top-tier tech companies don’t just look for developers; they look for problem solvers

--- temperature = 0.7 ---
To create an effective advertisement for LeetCode, you need to hit the "pain points" of a developer: the anxiety of the technical interview and the desire for a higher salary.

Here are three different approaches depending on the platform:

---

### Option 1: The "Career Advancement" Angle (Best for LinkedIn/Professional)
**Headline: Don’t just land the interview. Ace it.**

**Body:** 
The tech industry moves fast,

--- temperature = 1.0 ---
Here are a few options for a LeetCode advertisement, tailored to different pl

In [16]:
###############################################################################

#### Compare zero-shot vs few-shot prompting on the same task ##### 

client = genai.Client()

zero_shot = """
Classify the sentiment as Positive, Negative, or Neutral.
Text: "The app is useful, but it crashes too often."
Return only the label.
"""

few_shot = """
Classify the sentiment as Positive, Negative, or Neutral.
Return only the label.

Text: "This product saved me a lot of time."
Label: Positive

Text: "The interface is confusing and slow."
Label: Negative

Text: "The meeting starts at 3 PM."
Label: Neutral

Text: "The app is useful, but it crashes too often."
Label:
"""

for name, prompt in [
    ("Zero-shot", zero_shot),
    ("Few-shot", few_shot),
]:
    response = client.interactions.create(
        model="gemini-3.1-flash-lite",
        input=prompt,
        generation_config={"temperature": 0.2},
    )

    print(f"\n--- {name} ---")
    print(response.output_text)


--- Zero-shot ---
Negative

--- Few-shot ---
Negative


In [17]:
##### Prompt Design #####
client = genai.Client()
prompt_template = """
Role:
You are a senior Python engineer and LeetCode mentor.

Task:
Review the following Python solution. Identify bugs, explain the time and
space complexity, and suggest one improvement.

Constraints:
- Reply in Chinese.
- Do not rewrite the full solution unless there is a correctness bug.
- Keep the response under 200 words.
- Focus on correctness before optimization.

Output format:
1. Verdict: Correct / Incorrect
2. Bug or issue: <one sentence>
3. Complexity: Time O(...), Space O(...)
4. Improvement: <one actionable suggestion>

Code:
def containsDuplicate(nums):
    return len(nums) != len(set(nums))
"""


response = client.interactions.create(
    model="gemini-3.1-flash-lite",
    input=prompt_template,
)

print(response.output_text)

1. **Verdict**: Correct

2. **Bug or issue**: 无逻辑错误，该实现利用集合去重特性，准确判断是否存在重复元素。

3. **Complexity**: 
   - 时间复杂度：O(n)，构建集合需遍历数组一次。
   - 空间复杂度：O(n)，最坏情况下（无重复元素）集合需存储所有元素。

4. **Improvement**: 若对内存极其敏感且数组可能很大，可考虑使用“短路”逻辑：通过 `for` 循环配合 `set.add()` 提前判断，一旦发现重复元素立即返回 `True`，无需遍历整个数组，从而在平均情况下提升性能。
